In [10]:
# turn off pretty printing, because the slides can't handle the verticality
%pprint
# load supriya's ipython extension to capture audio/graphs
%load_ext supriya.ext.ipython

Pretty printing has been turned OFF
The supriya.ext.ipython extension is already loaded. To reload it, use:
  %reload_ext supriya.ext.ipython


In [11]:
import warnings

warnings.filterwarnings(action="default", module="supriya")

from supriya.scsynth import kill

kill()

In [12]:
# soundcheck

import supriya

server = supriya.Server()
server.

_IncompleteInputError: incomplete input (1248818067.py, line 6)

March 18th, 2025

# Supriya: a Python API for SuperCollider

Joséphine Wolf Oberholtzer (she/her) <br/>
https://josephine-wolf-oberholtzer.com/ <br/>
https://github.com/supriya-project/supriya/

## Preamble

### Who am I

- ~16 years of experience in Python
- PhD from Harvard in music
- Worked at Forced Exposure, Discogs, Capital One, Cortico/MIT

### What Supriya is for

- A foundational layer for application code
- Consistent API for realtime and non-realtime
- Consistent API for threaded and async concurrency models
- Reified server command API
- Musical-time-aware clocks
- Let's just make it easy to use SuperCollider inside the Python ecosystem!

### What Supriya isn't for

- Live coding
- UIs
- IDEs
- MIDI

### Why Python?

- Millions of people use it (visibility / support is good)
- Huge vibrant ecosystem (most things I want, somebody already invented and still support)
- (Relatively) simple syntax
- Good developer experience (tracebacks, debugging, testing, linting, type checking, CI, docs tooling, etc.)

### What I'm gonna cover

- Basic usage
- Design principles

### What I'm not gonna cover

- How to use Python
- How to use SuperCollider
- Installation

### Principles

- Avoid globals, avoid implicitness
- Make it ~boring~ simple (avoid multiple means to the same ends, resist flexibility, "there should be one (and preferably only one) obvious way to do it")
- Make it verbose (give everything names, avoid abbreviations, prefer keywords over positionals, avoid fanciful names, strive for using terms of art aligned with the wider language ecosystem)
- Make it similar (strive for similar if not identical means of interacting with similar things)
- Make it narrowly focused (don't have to _implement_ what's already implemented in the rest of the ecosystem, but may have to _integrate_ with it)
- Make it easily introspectable
- Make it easily testable

In [1]:
import this

The Zen of Python, by Tim Peters

Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex.
Complex is better than complicated.
Flat is better than nested.
Sparse is better than dense.
Readability counts.
Special cases aren't special enough to break the rules.
Although practicality beats purity.
Errors should never pass silently.
Unless explicitly silenced.
In the face of ambiguity, refuse the temptation to guess.
There should be one-- and preferably only one --obvious way to do it.
Although that way may not be obvious at first unless you're Dutch.
Now is better than never.
Although never is often better than *right* now.
If the implementation is hard to explain, it's a bad idea.
If the implementation is easy to explain, it may be a good idea.
Namespaces are one honking great idea -- let's do more of those!


## Realtime Servers

### Import supriya

In [13]:
# supriya is a package, so let's import it
import supriya

In [14]:
# everything in python is an object, so we can do some inspecting
supriya

<module 'supriya' from '/Users/josephine/Source/github.com/supriya-project/supriya/supriya/__init__.py'>

In [15]:
# let's look at the names defined inside the supriya namespace
dir(supriya)

['AddAction', 'AsyncClock', 'AsyncOfflineClock', 'AsyncServer', 'BaseClock', 'BaseServer', 'Buffer', 'BufferGroup', 'Bus', 'BusGroup', 'CalculationRate', 'Clock', 'ClockContext', 'Context', 'DoneAction', 'Envelope', 'Group', 'HeaderFormat', 'Node', 'OfflineClock', 'Options', 'OscBundle', 'OscCallback', 'OscMessage', 'ParameterRate', 'Pattern', 'SampleFormat', 'Score', 'Server', 'ServerLifecycleEvent', 'ServerShutdownEvent', 'Synth', 'SynthDef', 'SynthDefBuilder', 'TimeUnit', 'UGen', 'UGenOperable', 'UGenVector', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '__version_info__', '_version', 'assets', 'assets_path', 'clocks', 'contexts', 'conversions', 'default', 'enums', 'exceptions', 'ext', 'graph', 'io', 'osc', 'output_path', 'patterns', 'play', 'plot', 'render', 'sclang', 'scsynth', 'synthdef', 'typing', 'ugens', 'utils']

### Import `Server`

In [16]:
from supriya import Server

In [17]:
server = Server()

In [18]:
dir(server)

['__abstractmethods__', '__annotations__', '__class__', '__contains__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_add_node_to_children', '_add_request_with_completion', '_add_requests', '_allocate_id', '_apply_completions', '_audio_bus_allocator', '_boot_future', '_boot_status', '_buffer_allocator', '_buffers', '_client_id', '_contexts', '_control_bus_allocator', '_exit_future', '_free_id', '_get_allocator', '_get_moment', '_get_next_sync_id', '_get_options', '_get_request_context', '_handle_done_b_alloc', '_handle_done_b_alloc_read', '_handle_done_b_alloc_read_channel', '_handle_done_b_free', '_handle_fail', '_h

In [19]:
[name for name in dir(server) if not name.startswith("_")]

['add_buffer', 'add_buffer_group', 'add_bus', 'add_bus_group', 'add_group', 'add_synth', 'add_synthdefs', 'at', 'audio_input_bus_group', 'audio_output_bus_group', 'boot', 'boot_future', 'boot_status', 'clear_schedule', 'client_id', 'close_buffer', 'connect', 'copy_buffer', 'default_group', 'disconnect', 'do_nothing', 'dump_tree', 'exit_future', 'fill_buffer', 'fill_bus_range', 'free_all_synthdefs', 'free_buffer', 'free_buffer_group', 'free_bus', 'free_bus_group', 'free_group_children', 'free_node', 'free_synthdefs', 'generate_buffer', 'get_buffer', 'get_buffer_range', 'get_bus', 'get_bus_range', 'get_synth_control_range', 'get_synth_controls', 'is_owner', 'latency', 'load_synthdefs', 'load_synthdefs_directory', 'map_node', 'move_node', 'name', 'normalize_buffer', 'options', 'order_nodes', 'osc_protocol', 'pause_node', 'process_protocol', 'query_buffer', 'query_node', 'query_status', 'query_tree', 'query_version', 'quit', 'read_buffer', 'reboot', 'register_lifecycle_callback', 'register

In [5]:
server

<Server OFFLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>

### Server options

In [6]:
# TODO: Make Options a dataclass
server.options

Options(audio_bus_channel_count=1024, block_size=64, buffer_count=1024, control_bus_channel_count=16384, executable=None, hardware_buffer_size=None, initial_node_id=1000, input_bus_channel_count=8, input_device=None, input_stream_mask='', ip_address='127.0.0.1', load_synthdefs=True, maximum_logins=1, maximum_node_count=1024, maximum_synthdef_count=1024, memory_locking=False, memory_size=8192, output_bus_channel_count=8, output_device=None, output_stream_mask='', password=None, port=57110, protocol='udp', random_number_generator_count=64, realtime=True, restricted_path=None, safety_clip=None, sample_rate=None, threads=6, ugen_plugins_path=None, verbosity=0, wire_buffer_count=64, zero_configuration=False)

In [20]:
!scsynth -h

supercollider_synth  options:
   -v print the supercollider version and exit
   -u <udp-port-number>    a port number 0-65535
   -t <tcp-port-number>    a port number 0-65535
   -B <bind-to-address>
          Bind the UDP or TCP socket to this address.
          Default 127.0.0.1. Set to 0.0.0.0 to listen on all interfaces.
   -c <number-of-control-bus-channels> (default 16384)
   -a <number-of-audio-bus-channels>   (default 1024)
   -i <number-of-input-bus-channels>   (default 8)
   -o <number-of-output-bus-channels>  (default 8)
   -z <block-size>                     (default 64)
   -Z <hardware-buffer-size>           (default 0)
   -S <hardware-sample-rate>           (default 0)
   -b <number-of-sample-buffers>       (default 1024)
   -n <max-number-of-nodes>            (default 1024)
   -d <max-number-of-synth-defs>       (default 1024)
   -m <real-time-memory-size>          (default 8192)
   -w <number-of-wire-buffers>         (default 64)
   -r <number-of-random-seeds>         (d

### Boot the server

In [21]:
server.boot()

<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>

### Inspect the server

In [22]:
server.status

StatusInfo(actual_sample_rate=48000.26430658156, average_cpu_usage=0.026777375489473343, group_count=2, peak_cpu_usage=0.07574062794446945, synth_count=0, synthdef_count=32, target_sample_rate=48000.0, ugen_count=0)

In [23]:
print(tree := server.query_tree())

NODE TREE 0 group
    1 group


In [24]:
# this is actually a query tree object, not just a string
# ... which is helpful in more complex unit testing situations
# ... because the tree can be annotated with information beyond what scsynth provides
# ... but we can still use string comparisons
tree

QueryTreeGroup(node_id=0, annotation=None, children=[QueryTreeGroup(node_id=1, annotation=None, children=[])])

### Reset the server

In [ ]:
# without quitting...
# - delete all nodes and synthdefs
# - and clear all allocators
server.reset()

In [ ]:
print(server.query_tree())

### Quit the server

In [ ]:
server.quit()

### Boot with options

In [ ]:
server.boot(maximum_logins=2)

### Multiple users

In [ ]:
# We can create a handle to a second server proxy,
# pointed back at the same IP address and port as the first
other_server = Server()
other_server

In [ ]:
server  # note the port is the same as `other_server`

In [ ]:
other_server.connect()  # connect to the original server via .connect()

In [ ]:
print(f"{server.client_id=}")
print(f"{other_server.client_id=}")

In [ ]:
other_server.disconnect()  # disconnect from the original server

In [ ]:
server  # the original server remains online

n.b. it's not clear to me how to release a client ID once consumed

### Lifecycle events

In [ ]:
def on_event(event):
    print(event)

for event_type in supriya.ServerLifecycleEvent:
    server.on(event_type, on_event)
    other_server.on(event_type, on_event)

In [ ]:
server.reboot()

In [ ]:
# this will panic
other_server.boot()

In [ ]:
server.quit()

## Context Entities

### Groups

In [ ]:
server = Server().boot()

In [ ]:
# add a group
group = server.add_group()
group

In [ ]:
print(dir(group))

In [ ]:
print(server.query_tree())

In [ ]:
# add a group to the group
child_group = group.add_group()
print(server.query_tree())

In [ ]:
# move the child group into the default group
child_group.move(target_node=server.default_group, add_action="ADD_TO_TAIL")
print(server.query_tree())

In [ ]:
# free the original parent group
group.free()
print(server.query_tree())

### Synths

In [ ]:
# add a synth (this will fail, silently)
synth = child_group.add_synth(synthdef=supriya.default)

In [ ]:
# we have a proxy to a synth
synth

In [ ]:
# but nothing on the server, because the request failed
print(server.query_tree())

#### Completion (1) via lambda

In [ ]:
# using on_completion keyword argument
# this executes immediately
server.add_synthdefs(
    supriya.default,
    on_completion=lambda server: child_group.add_synth(synthdef=supriya.default),
) 
server.sync()
print(server.query_tree())

#### Completion (2) via context manager

In [ ]:
# using completion context manager
# executes inside a context moment (.at())
with server.at():
    with server.add_synthdefs(supriya.default):
        synth = child_group.add_synth(synthdef=supriya.default, frequency=666)
server.sync()
print(server.query_tree())

In [ ]:
synth.set(frequency=555)

In [ ]:
synth.free()

In [ ]:
# free the remaining synth in the child group by cascading a control setting
child_group.set(gate=0)

### Buses

In [ ]:
# add a control bus
control_bus = server.add_bus()
control_bus

In [ ]:
# add an audio bus
audio_bus = server.add_bus("audio")
audio_bus

In [ ]:
# add a bus group
control_bus_group = server.add_bus_group(count=4)
control_bus_group

In [ ]:
# a bus group aggregates together bus objects
for bus in control_bus_group:
    print(bus)

#### Default buses

In [25]:
server.audio_input_bus_group

BusGroup(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>, id_=8, calculation_rate=CalculationRate.AUDIO, count=8)

In [26]:
server.audio_output_bus_group

BusGroup(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>, id_=0, calculation_rate=CalculationRate.AUDIO, count=8)

#### Mapping

In [ ]:
# buses and bus groups all can be expressed as map symbols
for bus_like in (control_bus, audio_bus, control_bus_group):
    print(bus_like.map_symbol())

In [ ]:
# we can map buses to synth controls
synth.map(amplitude=control_bus, frequency=audio_bus)
print(server.query_tree())

In [17]:
gate_bus = server.add_bus()
gate_bus.set(1)
synth.map(gate=gate_bus)
gate_bus.set(0)

NameError: name 'synth' is not defined

In [ ]:
# we can release the bus IDs back to the allocator pools
for bus_like in (control_bus, audio_bus, control_bus_group):
    bus_like.free()

#### Shared Memory

In [9]:
another_bus_group = server.add_bus_group(count=4)

In [13]:
server._shm

In [16]:
server._shm[another_bus_group]

[0.10000000149011612, 0.5, 0.30000001192092896, 0.20000000298023224]

In [15]:
server._shm[another_bus_group] = [0.1, 0.5, 0.3, 0.2]

In [16]:
server._shm[another_bus_group]

[0.10000000149011612, 0.5, 0.30000001192092896, 0.20000000298023224]

### Buffers

- allocate and free
- read sound files
- generating
- plotting

In [ ]:
# i can't remember this one, so let's ask for help
help(server.add_buffer)

In [ ]:
# add a mono buffer with 64 frames
buffer = server.add_buffer(channel_count=1, frame_count=64)

In [ ]:
# query the buffer
buffer.query()

In [ ]:
# i can't remember this one either
help(buffer.generate)

In [ ]:
# generate a chebyshev polynomial in wavetable format
buffer.generate("cheby", amplitudes=[0.1, 0.2, 0.05], as_wavetable=True)

In [ ]:
# get values at indices in the buffer
buffer.get(0, 2, 4, 8)

In [ ]:
# get a range of values
buffer.get_range(index=0, count=16)

In [ ]:
# import rendering functions
from supriya import play, plot

In [ ]:
# plot the buffer
await plot(buffer)

In [ ]:
# create a second buffer and generate differently
other_buffer = server.add_buffer(frame_count=1024)
other_buffer.generate("sine1", amplitudes=[0.1, 0.3, 0.5])
await plot(other_buffer)

In [ ]:
# read a file into a buffer
file_path = supriya.assets_path / "audio/birds/birds-01.wav"
yet_another_buffer = server.add_buffer(file_path=file_path)
await plot(yet_another_buffer)

In [ ]:
# normally play() isn't async
# but for fussy reasons related to jupyter itself being async,
# we have to await here
await play(yet_another_buffer)

In [ ]:
# allocate a group of buffers, e.g.
buffer_group = server.add_buffer_group(count=4, frame_count=512, channel_count=1)
print(buffer_group)
for buffer_ in buffer_group:
    print(buffer_)

In [ ]:
# free all the buffers
buffer.free()
other_buffer.free()
yet_another_buffer.free()
buffer_group.free()

### Enums

In [ ]:
# calculation rates are expressed via an enum
from supriya import CalculationRate

for rate in sorted(CalculationRate):
    print(repr(rate), rate)

## Non-Realtime Scores

In [53]:
from supriya import Score, default, play

score = Score()
score

<Score [/Applications/SuperCollider.app/Contents/Resources/scsynth -N ...]>

In [54]:
# inspect the score's namespace
# note: no queries, only mutations
dir(score)

['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__render__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_add_request_with_completion', '_add_requests', '_allocate_id', '_apply_completions', '_audio_bus_allocator', '_boot_status', '_buffer_allocator', '_client_id', '_control_bus_allocator', '_free_id', '_get_allocator', '_get_moment', '_get_next_sync_id', '_get_options', '_get_request_context', '_latency', '_lock', '_name', '_node_id_allocator', '_options', '_pop_completion', '_pop_moment', '_push_completion', '_push_moment', '_requests', '_resolve_node', '_setup_allocators', '_sync_id', '_sync_id_maximum',

In [55]:
# add a synthdef at timestamp 0
with score.at(0):
    score.add_synthdefs(default)

# strum a series of synths
synths = []
for i in range(12):
    with score.at(i / 4):
        synths.append(score.add_synth(synthdef=default, frequency=111 * (i + 1)))

# free all of them
with score.at(3):
    for synth in synths:
        synth.free()

# pad out a no-op while the envelopes decay
with score.at(4):
    score.do_nothing()

In [56]:
# play the score (and capture into the notebook)
_ = await play(score)

In [31]:
# render the score, returning the path and exit code
await score.render()

(PosixPath('/Users/josephine/Library/Caches/supriya/score-b5e6166319aa99a4a61cb3ce6e5e79fda03de211129e3ea3043e3dfda894feb8.aiff'), 0)

In [32]:
# this will error!
# no queries, only mutations
with score.at(0):
    score.query_tree()

AttributeError: 'Score' object has no attribute 'query_tree'

In [57]:
# iterate the osc bundles in the score
for bundle in score.iterate_osc_bundles():
    print(repr(bundle))

OscBundle(timestamp=0.0, contents=[OscMessage('/d_recv', b'SCgf\x00\x00\x00\x02\x00\x01\x07default\x00\x00\x00\x0c\x00\x00\x00\x00>\x99\x99\x9a<#\xd7\n?333@\x00\x00\x00\xbe\xcc\xcc\xcd>\xcc\xcc\xcdEz\x00\x00E\x9c@\x00E\x1c@\x00EH\x00\x00?\x80\x00\x00\x00\x00\x00\x05=\xcc\xcc\xcdC\xdc\x00\x00?\x80\x00\x00?\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x05\tamplitude\x00\x00\x00\x00\tfrequency\x00\x00\x00\x01\x04gate\x00\x00\x00\x02\x03pan\x00\x00\x00\x03\x03out\x00\x00\x00\x04\x00\x00\x00\x14\x07Control\x01\x00\x00\x00\x00\x00\x00\x00\x04\x00\x00\x01\x01\x01\x01\x06VarSaw\x02\x00\x00\x00\x03\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x01\xff\xff\xff\xff\x00\x00\x00\x00\xff\xff\xff\xff\x00\x00\x00\x01\x02\x05Linen\x01\x00\x00\x00\x05\x00\x00\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x02\xff\xff\xff\xff\x00\x00\x00\x02\xff\xff\xff\xff\x00\x00\x00\x03\xff\xff\xff\xff\x00\x00\x00\x01\xff\xff\xff\xff\x00\x00\x00\x04\x01\x07Control\x00\x00\x00\x00\x00\x00\x00\x00\x01\x00\x04\x00\x04Rand\

In [58]:
# but actually we store an intermediate format... requests
for bundle in score.iterate_request_bundles():
    print(bundle)

RequestBundle(contents=[ReceiveSynthDefs(synthdefs=(<SynthDef: default>,), on_completion=None), NewSynth(synthdef=<SynthDef: default>, synth_id=1000, add_action=AddAction.ADD_TO_HEAD, target_node_id=0, controls={'frequency': 111.0})], timestamp=0.0)
RequestBundle(contents=[NewSynth(synthdef=<SynthDef: default>, synth_id=1001, add_action=AddAction.ADD_TO_HEAD, target_node_id=0, controls={'frequency': 222.0})], timestamp=0.25)
RequestBundle(contents=[NewSynth(synthdef=<SynthDef: default>, synth_id=1002, add_action=AddAction.ADD_TO_HEAD, target_node_id=0, controls={'frequency': 333.0})], timestamp=0.5)
RequestBundle(contents=[NewSynth(synthdef=<SynthDef: default>, synth_id=1003, add_action=AddAction.ADD_TO_HEAD, target_node_id=0, controls={'frequency': 444.0})], timestamp=0.75)
RequestBundle(contents=[NewSynth(synthdef=<SynthDef: default>, synth_id=1004, add_action=AddAction.ADD_TO_HEAD, target_node_id=0, controls={'frequency': 555.0})], timestamp=1.0)
RequestBundle(contents=[NewSynth(syn

## OSC

In [ ]:
server.reset()

In [ ]:
server.osc_protocol

In [ ]:
with server.osc_protocol.capture() as transcript:
    with server.at():
        with server.add_synthdefs(supriya.default):
            group = server.add_group()
            synth = group.add_synth(synthdef=supriya.default, frequency=666)

In [ ]:
for x in transcript: print(x)

### OSC messages & bundles

### Sending messages

In [ ]:
server.send([])

### OSC protocols

In [ ]:
server.osc_protocol

### Capturing IO

In [ ]:
with server.osc_protocol.capture() as transcript:
    with server.at():
        group = server.add_group()
        with server.add_synthdefs(supriya.default):
            synth = group.add_synth(synthdef=supriya.default, frequency=666)
    server.sync()

In [ ]:
for entry in transcript:
    print(entry)

### Callbacks

### Moments & completions

In [19]:
# .at() gives us a Moment
server.at()

Moment(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>, seconds=None, closed=False, requests=[])

In [20]:
# Moments can have specific timestamps, relative to real time
import time

server.at(time.time() + 1)

Moment(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>, seconds=1740155625.003901, closed=False, requests=[])

In [27]:
# we can issue multiple commands inside the same moment
with server.osc_protocol.capture() as transcript:
    with server.at() as moment:
        group_a = server.add_group()
        group_b = server.add_group()
        group_a.add_synth(synthdef=supriya.default)
        group_b.add_group()

In [28]:
moment

Moment(context=<Server ONLINE [/Applications/SuperCollider.app/Contents/Resources/scsynth -R 0 -l 1 -u 57110]>, seconds=None, closed=True, requests=[(NewGroup(items=[(1006, AddAction.ADD_TO_HEAD, 1)]), None), (NewGroup(items=[(1007, AddAction.ADD_TO_HEAD, 1)]), None), (NewSynth(synthdef=<SynthDef: default>, synth_id=1008, add_action=AddAction.ADD_TO_HEAD, target_node_id=1006, controls={}), None), (NewGroup(items=[(1009, AddAction.ADD_TO_HEAD, 1007)]), None)])

In [29]:
# moment aggregates request objects
for request in moment.requests:
    print(request)

(NewGroup(items=[(1006, AddAction.ADD_TO_HEAD, 1)]), None)
(NewGroup(items=[(1007, AddAction.ADD_TO_HEAD, 1)]), None)
(NewSynth(synthdef=<SynthDef: default>, synth_id=1008, add_action=AddAction.ADD_TO_HEAD, target_node_id=1006, controls={}), None)
(NewGroup(items=[(1009, AddAction.ADD_TO_HEAD, 1007)]), None)


In [30]:
# when serialized to osc, some requests are coalesced
# note the first /g_new includes two group definitions
for x in transcript:
    print(x)

CaptureEntry(timestamp=1740155804.657043, label='S', message=OscBundle(contents=[OscMessage('/g_new', 1006, 0, 1, 1007, 0, 1), OscMessage('/s_new', 'default', 1008, 0, 1006), OscMessage('/g_new', 1009, 0, 1007)]))


### Requests & responses

In [ ]:
import supriya.contexts.requests

dir(supriya.contexts.requests)

## Synthdefs

### Building SynthDefs (1)

In [ ]:
r"""
SynthDef(\simple, { arg out=0, freq=440;
	Out.ar(out, SinOsc.ar(freq));
});
"""

In [ ]:
# A simple SynthDef using the builder pattern
from supriya.ugens import SynthDefBuilder
from supriya.ugens import Out, SinOsc

with SynthDefBuilder(freq=440, out=0) as builder:
    source = SinOsc.ar(frequency=builder["freq"])
    Out.ar(bus=builder["out"], source=[source, source])

simple_synthdef = builder.build(name="simple")
simple_synthdef

### Building SynthDefs (2)

In [ ]:
r"""
*makeDefaultSynthDef {
    SynthDef(\default, { arg out=0, freq=440, amp=0.1, pan=0, gate=1;
        var z;
        z = LPF.ar(
            Mix.new(VarSaw.ar(freq + [0, Rand(-0.4,0.0), Rand(0.0,0.4)], 0, 0.3, 0.3)),
            XLine.kr(Rand(4000,5000), Rand(2500,3200), 1)
        ) * Linen.kr(gate, 0.01, 0.7, 0.3, 2);
        OffsetOut.ar(out, Pan2.ar(z, pan, amp));
    }, [\ir]).add;
}
"""

In [ ]:
# More imports
from supriya.enums import DoneAction, ParameterRate
from supriya.ugens import (
    LPF,
    Linen,
    Mix,
    OffsetOut,
    Pan2,
    Parameter,
    Rand,
    SynthDefBuilder,
    VarSaw,
    XLine,
)

In [ ]:
# Define a builder
builder = SynthDefBuilder(
    out=Parameter(rate=ParameterRate.SCALAR, value=0),
    amplitude=0.1,
    frequency=440,
    gate=1,
    pan=0.5,
)

In [ ]:
# Use the builder as a context manager
with builder:
    linen = Linen.kr(
        attack_time=0.01,
        done_action=DoneAction.FREE_SYNTH,
        gate=builder["gate"],
        release_time=0.3,
        sustain_level=0.7,
    )

In [ ]:
# Use the builder again
with builder:
    low_pass = LPF.ar(
        source=Mix.new(
            VarSaw.ar(
                frequency=builder["frequency"]
                + (
                    0,
                    Rand.ir(minimum=-0.4, maximum=0.0),
                    Rand.ir(minimum=0.0, maximum=0.4),
                ),
                width=0.3,
            )
        )
        * 0.3,
        frequency=XLine.kr(
            start=Rand.ir(minimum=4000, maximum=5000),
            stop=Rand.ir(minimum=2500, maximum=3200),
        ),
    )

In [ ]:
# And again and again
with builder:
    panner = Pan2.ar(
        source=low_pass * linen * builder["amplitude"], position=builder["pan"]
    )

with builder:
    OffsetOut.ar(bus=builder["out"], source=panner)

In [ ]:
default = builder.build(name="default")
default

### The `synthdef` decorator

In [ ]:
# N.B. I'm not fond of this one because of
# a) how magical it is (not very, but just enough) but mainly because
# b) it makes type-checking difficult
from supriya.ugens import synthdef

In [ ]:
@synthdef("ir")
def default_decorated(out=0, amplitude=0.1, frequency=440, gate=1, pan=0.5):
    linen = Linen.kr(
        attack_time=0.01,
        done_action=DoneAction.FREE_SYNTH,
        gate=gate,
        release_time=0.3,
        sustain_level=0.7,
    )
    low_pass = LPF.ar(
        source=Mix.new(
            VarSaw.ar(
                frequency=frequency
                + (
                    0,
                    Rand.ir(minimum=-0.4, maximum=0.0),
                    Rand.ir(minimum=0.0, maximum=0.4),
                ),
                width=0.3,
            )
        )
        * 0.3,
        frequency=XLine.kr(
            start=Rand.ir(minimum=4000, maximum=5000),
            stop=Rand.ir(minimum=2500, maximum=3200),
        ),
    )
    panner = Pan2.ar(
        source=low_pass * linen * amplitude, position=pan
    )
    _ = OffsetOut.ar(bus=out, source=panner)

In [ ]:
default_decorated

In [ ]:
from supriya.ugens import Out, SinOsc

@synthdef()
def foo(out=0):
    print(repr(out))
    _ = Out.ar(source=SinOsc.kr())

foo

### Graphing SynthDefs

In [ ]:
print(default)

In [ ]:
from supriya import graph

_ = graph(default)

### SynthDef internals

In [ ]:
dir(default)

In [ ]:
default.name

In [ ]:
default.anonymous_name

In [ ]:
default.effective_name

In [ ]:
default.parameters

In [ ]:
default.ugens

In [ ]:
default.has_gate

### UGen methods

In [ ]:
dir(SinOsc)

### SynthDef (de)compilation

In [ ]:
# SynthDefs compile to byte strings
compiled = default.compile()
compiled

In [ ]:
# valid byte strings can be decompiled back into SynthDefs
from supriya.ugens import decompile_synthdef

decompiled = decompile_synthdef(compiled)
decompiled

In [ ]:
# sanity-check: the decompiled SynthDef is not the same in memory
default is decompiled

### Compiling via sclang

In [ ]:
# Supriya provides utilities for compiling via sclang.
# This is intended for validating its own logic vs sclang (as a reference spec).
from supriya.ugens import SuperColliderSynthDef

sc_synthdef = SuperColliderSynthDef(
    "foo", "Out.ar(0, SinOsc.ar(freq: 420) * SinOsc.ar(freq: 440))"
)
sc_compiled_synthdef = sc_synthdef.compile()  # return bytes
sc_compiled_synthdef

In [ ]:
# The sclang-derived SynthDef byte string can be decompiled back into a SynthDef.
print(decompile_synthdef(sc_compiled_synthdef))

### UGen metaprogramming

In [ ]:
from supriya.ugens import UGen, param, ugen

# A dupe of SinOsc
@ugen(ar=True, kr=True, is_pure=True)
class AnotherSinOsc(UGen):
    frequency = param(440.0)
    phase = param(0.0)

In [ ]:
AnotherSinOsc.ar()

In [ ]:
AnotherSinOsc.kr()

In [ ]:
# This won't work because ir=True wasn't set
AnotherSinOsc.ir()

In [ ]:
# A dupe of Out
@ugen(ar=True, kr=True, is_output=True, channel_count=0, fixed_channel_count=True)
class AnotherOut(UGen):
    bus = param(0)
    source = param(unexpanded=True)

AnotherOut.ar(source=AnotherSinOsc.ar())

In [ ]:
from supriya.ugens.pv import PV_ChainUGen

# A dupe of PV_BinShift
@ugen(kr=True, is_width_first=True)
class AnotherPV_BinShift(PV_ChainUGen):
    pv_chain = param()
    stretch = param(1.0)
    shift = param(0.0)
    interpolate = param(0)

# This won't work because of missing pv_chain argument
AnotherPV_BinShift.kr()

In [ ]:
"""
def ugen(
    *,
    ar: bool = False,
    kr: bool = False,
    ir: bool = False,
    dr: bool = False,
    new: bool = False,
    has_done_flag: bool = False,
    is_input: bool = False,
    is_multichannel: bool = False,
    is_output: bool = False,
    is_pure: bool = False,
    is_width_first: bool = False,
    channel_count: int = 1,
    fixed_channel_count: bool = False,
    signal_range: Optional[int] = None,
) -> Callable[[Type["UGen"]], Type["UGen"]]:
    ...
"""

## Clocks & Patterns

## Concurrency

- server
- async server
- threaded concurrency
- event loop / coroutine concurrency

## Contexts

- Mutation interface (all contexts)
- Query interface (realtime only)
- Mutations are realtime/nonrealtime agnostic
- Mutations are concurrency agnostic

## Cleanliness

- docs
- typing
- testing
- ci/cd

## Epilogue

Questions?

Thanks, darlings! <br/>
xoxo, joséphine

https://josephine-wolf-oberholtzer.com/ <br/>
https://github.com/supriya-project/supriya/

### PS: What's in a name